<a href="https://colab.research.google.com/github/visaodosdados/Projeto_01_Panorama_Ecommerce/blob/main/02_Limpeza_e_Preparacao.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Limpeza e Preparação dos Dados

Objetivo:

Transformar os dados brutos do Olist em uma base analítica consistente para análise exploratória e construção de dashboards.

Importar bibliotecas:

In [1]:
import pandas as pd
import numpy as np

Conectar diretório no drive:

In [2]:
from google.colab import drive

drive.mount('/content/drive')

Mounted at /content/drive


# Carregar os dados

In [3]:
BASE_PATH = "/content/drive/MyDrive/Visão dos Dados/Portfolio/Projeto_01_Panorama_Ecommerce"

RAW_PATH = f"{BASE_PATH}/01_Dados_Brutos"
TREATED_PATH = f"{BASE_PATH}/02_Dados_Tratados"

In [4]:
orders = pd.read_csv(f"{RAW_PATH}/olist_orders_dataset.csv")
customers = pd.read_csv(f"{RAW_PATH}/olist_customers_dataset.csv")
items = pd.read_csv(f"{RAW_PATH}/olist_order_items_dataset.csv")
payments = pd.read_csv(f"{RAW_PATH}/olist_order_payments_dataset.csv")
products = pd.read_csv(f"{RAW_PATH}/olist_products_dataset.csv")

# Converter datas

In [5]:
orders.columns

Index(['order_id', 'customer_id', 'order_status', 'order_purchase_timestamp',
       'order_approved_at', 'order_delivered_carrier_date',
       'order_delivered_customer_date', 'order_estimated_delivery_date'],
      dtype='object')

In [6]:
colunas_data = [
    'order_purchase_timestamp',
    'order_approved_at',
    'order_delivered_carrier_date',
    'order_delivered_customer_date',
    'order_estimated_delivery_date'
]

for coluna in colunas_data:
    orders[coluna] = pd.to_datetime(
        orders[coluna],
        errors='coerce'
    )

In [7]:
orders.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 99441 entries, 0 to 99440
Data columns (total 8 columns):
 #   Column                         Non-Null Count  Dtype         
---  ------                         --------------  -----         
 0   order_id                       99441 non-null  object        
 1   customer_id                    99441 non-null  object        
 2   order_status                   99441 non-null  object        
 3   order_purchase_timestamp       99441 non-null  datetime64[ns]
 4   order_approved_at              99281 non-null  datetime64[ns]
 5   order_delivered_carrier_date   97658 non-null  datetime64[ns]
 6   order_delivered_customer_date  96476 non-null  datetime64[ns]
 7   order_estimated_delivery_date  99441 non-null  datetime64[ns]
dtypes: datetime64[ns](5), object(3)
memory usage: 6.1+ MB


# Verificar valores nulos

In [8]:
orders.isna().sum().sort_values(
    ascending=False
)

,0
order_delivered_customer_date,2965
order_delivered_carrier_date,1783
order_approved_at,160
order_id,0
order_purchase_timestamp,0
order_status,0
customer_id,0
order_estimated_delivery_date,0


In [9]:
products.isna().sum().sort_values(
    ascending=False
)

,0
product_category_name,610
product_description_lenght,610
product_name_lenght,610
product_photos_qty,610
product_weight_g,2
product_height_cm,2
product_length_cm,2
product_width_cm,2
product_id,0


# Criar variáveis de tempo

In [10]:
orders['ano'] = (
    orders['order_purchase_timestamp']
    .dt.year
)

orders['mes'] = (
    orders['order_purchase_timestamp']
    .dt.month
)

orders['data'] = (
    orders['order_purchase_timestamp']
    .dt.date
)

# Analisar duplicatas


In [11]:
orders.duplicated().sum()

np.int64(0)

In [12]:
customers.duplicated().sum()

np.int64(0)

In [13]:
items.duplicated().sum()

np.int64(0)

# Construir base analítica

Pedidos e clientes

In [14]:
base = orders.merge(
    customers,
    on='customer_id',
    how='left'
)

Adicionar items

In [15]:
base = base.merge(
    items,
    on='order_id',
    how='left'
)

Adicionar pagamentos

In [16]:
base = base.merge(
    payments,
    on='order_id',
    how='left'
)

# Verificar a Base

In [17]:
base.shape

(118434, 25)

In [18]:
base.head()

,order_id,customer_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date,ano,mes,...,order_item_id,product_id,seller_id,shipping_limit_date,price,freight_value,payment_sequential,payment_type,payment_installments,payment_value
0,e481f51cbdc54678b7cc49136f2d6af7,9ef432eb6251297304e76186b10a928d,delivered,2017-10-02 10:56:33,2017-10-02 11:07:15,2017-10-04 19:55:00,2017-10-10 21:25:13,2017-10-18,2017,10,...,1.0,87285b34884572647811a353c7ac498a,3504c0cb71d7fa48d967e0e4c94d59d9,2017-10-06 11:07:15,29.99,8.72,1.0,credit_card,1.0,18.12
1,e481f51cbdc54678b7cc49136f2d6af7,9ef432eb6251297304e76186b10a928d,delivered,2017-10-02 10:56:33,2017-10-02 11:07:15,2017-10-04 19:55:00,2017-10-10 21:25:13,2017-10-18,2017,10,...,1.0,87285b34884572647811a353c7ac498a,3504c0cb71d7fa48d967e0e4c94d59d9,2017-10-06 11:07:15,29.99,8.72,3.0,voucher,1.0,2.00
2,e481f51cbdc54678b7cc49136f2d6af7,9ef432eb6251297304e76186b10a928d,delivered,2017-10-02 10:56:33,2017-10-02 11:07:15,2017-10-04 19:55:00,2017-10-10 21:25:13,2017-10-18,2017,10,...,1.0,87285b34884572647811a353c7ac498a,3504c0cb71d7fa48d967e0e4c94d59d9,2017-10-06 11:07:15,29.99,8.72,2.0,voucher,1.0,18.59
3,53cdb2fc8bc7dce0b6741e2150273451,b0830fb4747a6c6d20dea0b8c802d7ef,delivered,2018-07-24 20:41:37,2018-07-26 03:24:27,2018-07-26 14:31:00,2018-08-07 15:27:45,2018-08-13,2018,7,...,1.0,595fac2a385ac33a80bd5114aec74eb8,289cdb325fb7e7f891c38608bf9e0962,2018-07-30 03:24:27,118.70,22.76,1.0,boleto,1.0,141.46
4,47770eb9100c2d0c44946d9cf07ec65d,41ce2a54c0b03bf3443c3d931a367089,delivered,2018-08-08 08:38:49,2018-08-08 08:55:23,2018-08-08 13:50:00,2018-08-17 18:06:29,2018-09-04,2018,8,...,1.0,aa4383b373c6aca5d8797843e5594415,4869f7a5dfa277a7dca6462dcf3b52b2,2018-08-13 08:55:23,159.90,19.22,1.0,credit_card,3.0,179.12


In [19]:
base.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 118434 entries, 0 to 118433
Data columns (total 25 columns):
 #   Column                         Non-Null Count   Dtype         
---  ------                         --------------   -----         
 0   order_id                       118434 non-null  object        
 1   customer_id                    118434 non-null  object        
 2   order_status                   118434 non-null  object        
 3   order_purchase_timestamp       118434 non-null  datetime64[ns]
 4   order_approved_at              118258 non-null  datetime64[ns]
 5   order_delivered_carrier_date   116360 non-null  datetime64[ns]
 6   order_delivered_customer_date  115037 non-null  datetime64[ns]
 7   order_estimated_delivery_date  118434 non-null  datetime64[ns]
 8   ano                            118434 non-null  int32         
 9   mes                            118434 non-null  int32         
 10  data                           118434 non-null  object        
 11  

# Criar Métricas


Receita:

In [20]:
base['receita_total'] = (
    base['price'] +
    base['freight_value']
)

Prazo de entrega:

In [21]:
base['dias_entrega'] = (
    base['order_delivered_customer_date']
    - base['order_purchase_timestamp']
).dt.days

#Validar KPIs

Receita total:

In [22]:
base['receita_total'].sum()

np.float64(16566687.309999999)

Número de pedidos:


In [23]:
base['order_id'].nunique()

99441

Número de clientes:

In [24]:
base['customer_unique_id'].nunique()

96096

# Salvar base tratada

In [25]:
base.to_csv(
    f"{TREATED_PATH}/base_analitica_olist.csv",
    index=False,
    encoding='utf-8-sig'
)